# Path Forms in `pygeoinf3D`

This notebook introduces **path-integral linear forms** — a key building block for modelling measurements that average a function along a curve (travel-time tomography, seismic ray integrals, etc.).

We cover:
1. **`Path3D`** — constructing curves from polylines, smooth callables, and point samples.
2. **`PathIntegralFunctional`** — evaluating $\ell_\gamma(f) = \int_\gamma f\,\mathrm{d}s$ on 3-D functions.
3. **Scalar algebra** — adding and scaling path-integral forms.
4. **`SumLinearForm`** — representing sums of forms with different paths.
5. **`SOLAOperator3D` with path forms** — stacking multiple path integrals into a forward operator.
6. **Sobolev representers** — computing $r_\ell$ such that $\ell(f) = \langle r_\ell, f \rangle_H$ via `Sobolev3D.from_dual`.
7. **End-to-end minimum-norm inversion** — recovering a model from synthetic path-integral data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from pygeoinf3d.geometries.box.domain import BoxDomain
from pygeoinf3d.spaces.lebesgue import Lebesgue3D
from pygeoinf3d.core.functions import Function3D
from pygeoinf3d.core.paths import Path3D

np.random.seed(42)

---
## 1. Building paths with `Path3D`

`Path3D` normalises every curve to a **unit arc-length parameterisation**: `path(t)` with $t \in [0, 1]$ always gives a point at fractional arc length $t$ along the curve.  Three construction modes are available.

In [ ]:
# ── Mode 1: straight line from polyline waypoints ─────────────────────────
path_straight = Path3D.from_polyline(np.array([
    [0.1, 0.5, 0.5],
    [0.9, 0.5, 0.5],
]))

# ── Mode 2: smooth callable (helical path) ────────────────────────────────
def helix(t):
    """t ∈ [0, 2π]  →  (x, y, z) on a helix inside the unit cube."""
    t = np.asarray(t, dtype=float)
    x = 0.5 + 0.35 * np.cos(t)
    y = 0.5 + 0.35 * np.sin(t)
    z = t / (2 * np.pi)
    if t.ndim == 0:
        return np.array([float(x), float(y), float(z)])
    return np.column_stack([x, y, z])

path_helix = Path3D.from_callable(helix, 0, 2 * np.pi)

# ── Mode 3: from_points (piecewise-linear through sample points) ──────────
ts_sample = np.linspace(0, 1, 10)
pts_sample = np.column_stack([
    ts_sample * 0.8 + 0.1,
    0.5 + 0.3 * np.sin(np.pi * ts_sample),
    np.full_like(ts_sample, 0.5),
])
path_pts = Path3D.from_points(pts_sample)

print(f"Straight path  arclength = {path_straight.arclength:.4f}")
print(f"Helix path     arclength = {path_helix.arclength:.4f}")
print(f"Points path    arclength = {path_pts.arclength:.4f}")

In [ ]:
from pygeoinf3d.plot import plot_paths3d_interactive

fig = plot_paths3d_interactive(
    [path_straight, path_helix, path_pts],
    labels=["Polyline (straight)", "Callable (helix)", "Points (sine arc)"],
    n_samples=200,
    show_markers=True,
    show_plot=True,
    title="Path3D constructions — interactive 3D view",
)

---
## 2. `PathIntegralFunctional` — evaluating $\ell_\gamma(f)$

A `PathIntegralFunctional` is a **linear form** on a Sobolev space $H$ over the domain $\Omega$.
In 3D, path integrals along 1-D curves are not bounded on plain $L^2(\Omega)$, so `PathIntegralFunctional` requires a `Sobolev3D` domain rather than a `Lebesgue3D`.

The evaluation is

$$\ell_\gamma(f) = \int_\gamma f\,\mathrm{d}s = |\gamma| \int_0^1 f(\gamma(t))\,\mathrm{d}t$$

where $|\gamma|$ is the arc length and the integral is approximated with the composite trapezoidal rule over `n_quad` nodes.

For sections 2–5 we use a simple identity-weight `Sobolev3D`, which is enough for forward evaluation. In section 6 we switch to a spectral `BesselSobolevBox` weight so that `Sobolev3D.from_dual` can compute representers.

### Analytic check — constant function

For $f \equiv c$ we have $\ell_\gamma(f) = c \cdot |\gamma|$, giving a simple verification.


In [ ]:
from pygeoinf3d.spaces.forms import PathIntegralFunctional
from pygeoinf.linear_operators import LinearOperator
from pygeoinf3d.spaces.sobolev import Sobolev3D

box = BoxDomain(np.zeros(3), np.ones(3))
space = Lebesgue3D(box)

# Identity-weight Sobolev space for forward evaluations in sections 2–5.
def _identity_map(f):
    return f

identity_op = LinearOperator(space, space, _identity_map, adjoint_mapping=_identity_map)
sobolev = Sobolev3D(space, identity_op, identity_op)

# ── Constant function f ≡ 2.5 ─────────────────────────────────────────────
const_val = 2.5
f_const = Function3D(box, evaluate_callable=lambda x: (
    np.full(len(x), const_val) if np.asarray(x).ndim == 2 else const_val
))

for path, name in [(path_straight, "straight"), (path_helix, "helix"), (path_pts, "arc")]:
    form = PathIntegralFunctional(sobolev, path, n_quad=500)
    val = form(f_const)
    exact = const_val * path.arclength
    print(f"{name:10s}  ℓ(f) = {val:.6f}   exact = {exact:.6f}   "
          f"rel-err = {abs(val - exact) / exact:.2e}")


### Quadrature convergence

How does the trapezoidal rule error decay as `n_quad` increases?

In [ ]:
# ── Smooth test function: f(x,y,z) = sin(πx)sin(πy)sin(πz) ───────────────
def f_smooth_callable(x):
    x = np.asarray(x, dtype=float)
    if x.ndim == 1:
        return float(np.prod(np.sin(np.pi * x)))
    return np.prod(np.sin(np.pi * x), axis=1)

f_smooth = Function3D(box, evaluate_callable=f_smooth_callable)

# Use a fine quadrature as "reference" value
ref_form = PathIntegralFunctional(sobolev, path_helix, n_quad=4000)
ref_val = ref_form(f_smooth)

n_quads = [2, 5, 10, 20, 50, 100, 200, 500, 1000]
errors = [abs(PathIntegralFunctional(sobolev, path_helix, n_quad=n)(f_smooth) - ref_val)
          for n in n_quads]

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.loglog(n_quads, errors, "o-", color="steelblue")
ax.set_xlabel("n_quad")
ax.set_ylabel("|error|")
ax.set_title("Trapezoidal quadrature convergence\n(helix path, smooth function)")
ax.grid(True, which="both", ls="--", alpha=0.4)
plt.tight_layout()
plt.show()


---
## 3. Scalar algebra on path forms

`PathIntegralFunctional` supports the standard linear-form algebra: `*`, `-`, `+` (same path only).

In [ ]:
form1 = PathIntegralFunctional(sobolev, path_straight, n_quad=200)
f_val = form1(f_smooth)

form_scaled = 3.0 * form1
form_neg = -form1
form_sum = form1 + form1   # same path → adds scales

print(f"ℓ(f)           = {f_val:.6f}")
print(f"(3·ℓ)(f)       = {form_scaled(f_smooth):.6f}   expected = {3 * f_val:.6f}")
print(f"(-ℓ)(f)        = {form_neg(f_smooth):.6f}    expected = {-f_val:.6f}")
print(f"(ℓ + ℓ)(f)     = {form_sum(f_smooth):.6f}     expected = {2 * f_val:.6f}")


---
## 4. `SumLinearForm` — sums over *different* paths

Adding two `PathIntegralFunctional` objects with **different** paths is not directly supported by `+` (which requires the same path instance). Instead, the dual mapping of `SOLAOperator3D` returns a `SumLinearForm` that stores weighted `(coefficient, form)` pairs and evaluates as

$$\ell(f) = \sum_i c_i\,\ell_i(f)$$

You can also construct `SumLinearForm` directly, which is useful for testing the representer logic.

In [ ]:
from pygeoinf3d.spaces.forms import SumLinearForm

form_straight = PathIntegralFunctional(sobolev, path_straight, n_quad=200)
form_pts = PathIntegralFunctional(sobolev, path_pts, n_quad=200)

# Weighted sum: 0.6 · ℓ_straight + 0.4 · ℓ_arc
sum_form = SumLinearForm(sobolev, [(0.6, form_straight), (0.4, form_pts)])

manual = 0.6 * form_straight(f_smooth) + 0.4 * form_pts(f_smooth)
via_sum = sum_form(f_smooth)

print(f"0.6·ℓ_straight(f) + 0.4·ℓ_arc(f)  = {manual:.8f}")
print(f"SumLinearForm(f)                   = {via_sum:.8f}")
print(f"Difference: {abs(via_sum - manual):.2e}")


---
## 5. `SOLAOperator3D` with path-integral forms

`SOLAOperator3D` stacks $m$ linear forms into a single forward operator

$$A : V \to \mathbb{R}^m, \qquad (Af)_i = \ell_{\gamma_i}(f)$$

This is the standard structure of a geophysical tomography forward problem (each path gives one travel-time measurement).

In [ ]:
from pygeoinf import EuclideanSpace
from pygeoinf3d.operators.sola import SOLAOperator3D

# Build 8 horizontal paths at different y-positions (a fan of parallel rays)
n_paths = 8
y_positions = np.linspace(0.15, 0.85, n_paths)

paths = [
    Path3D.from_polyline(np.array([[0.05, y, 0.5], [0.95, y, 0.5]]))
    for y in y_positions
]
forms = [PathIntegralFunctional(sobolev, p, n_quad=300) for p in paths]

G = SOLAOperator3D(sobolev, EuclideanSpace(n_paths), forms)

# True model: two Gaussian blobs
def true_model(x):
    x = np.asarray(x, dtype=float)
    single = x.ndim == 1
    pts = x[None, :] if single else x
    c1 = np.array([0.3, 0.4, 0.5])
    c2 = np.array([0.7, 0.65, 0.5])
    v = (1.5 * np.exp(-np.sum((pts - c1)**2, axis=1) / (2 * 0.1**2))
         - 1.0 * np.exp(-np.sum((pts - c2)**2, axis=1) / (2 * 0.12**2)))
    return float(v[0]) if single else v

u_true = Function3D(box, evaluate_callable=true_model, name="u_true")

# Synthetic measurements
d = G(u_true)

fig, ax = plt.subplots(figsize=(5, 3))
ax.barh(y_positions, d, height=0.06, color="steelblue", edgecolor="k", linewidth=0.5)
ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("$\ell_{\gamma_i}(u_{\mathrm{true}})$")
ax.set_ylabel("path y-position")
ax.set_title("Synthetic path-integral measurements")
plt.tight_layout()
plt.show()


In [ ]:
# Visualise the 8 measurement paths in 3D
fig = plot_paths3d_interactive(
    paths,
    labels=[f"path y={y:.2f}" for y in y_positions],
    n_samples=80,
    show_markers=True,
    title="8 measurement paths for SOLAOperator3D",
)

---
## 6. Sobolev representers via `Sobolev3D.from_dual`

Given a linear form $\ell_\gamma$ and a Sobolev space $H = (L^2, W)$ with inner product $\langle f, g\rangle_H = \langle Wf, g\rangle_{L^2}$, the **representer** $r_\gamma \in H$ satisfying

$$\ell_\gamma(f) = \langle r_\gamma, f\rangle_H \qquad \forall f \in H$$

is computed by `Sobolev3D.from_dual(ℓ)` using the spectral expansion

$$r_\gamma(\mathbf{x}) = \sum_{k=0}^{N-1} \ell_\gamma(\Phi_k)\,\nu_k^{W^{-1}}\,\Phi_k(\mathbf{x})$$

where $(\Phi_k, \nu_k^{W^{-1}})$ are the eigenfunctions / eigenvalues of $W^{-1}$ (here `BesselSobolevBox` with negative order).

A representer visualised as a 2-D slice gives the "resolution kernel" of the measurement — a spatially-smeared version of the path.

In [ ]:
from pygeoinf3d.providers.tensor_product import BoxLaplacianSpectrumProvider
from pygeoinf3d.operators.bessel_sobolev_box import BesselSobolevBox
from pygeoinf3d.spaces.sobolev import Sobolev3D

# Replace the identity-weight Sobolev space with a spectral one.
# This is required for Sobolev3D.from_dual to compute path-form representers.
provider = BoxLaplacianSpectrumProvider(box, truncation=(8, 8, 8), bc_type="neumann")
W = BesselSobolevBox(space, provider, shift=1.0, order=1.0)
W_inv = BesselSobolevBox(space, provider, shift=1.0, order=-1.0)
sobolev = Sobolev3D(space, W, W_inv)

# Representer for the middle path (y ≈ 0.5)
mid_form = PathIntegralFunctional(sobolev, paths[n_paths // 2], n_quad=300)
r = sobolev.from_dual(mid_form)

print("Representer identity check:")
print(f"  ℓ(r)        = {mid_form(r):.6f}")
print(f"  ‖r‖²_H      = {sobolev.inner_product(r, r):.6f}")


In [ ]:
from pygeoinf3d.plot import plot_function3d_isosurface

# Show the representer as a 3D isosurface — reveals the "smeared ray" shape
fig = plot_function3d_isosurface(
    r,
    resolution=30,
    n_levels=4,
    sigma=0.5,
    opacity=0.4,
    colorscale="RdBu_r",
    title=f"Sobolev representer for path y={y_positions[n_paths // 2]:.2f} — isosurface",
)

In [ ]:
from pygeoinf3d.plot import compare_function3d_axis_sliders

# Compute representers for 4 paths spanning the fan
repr_indices = [1, 3, 5, 7]
representers = [
    sobolev.from_dual(PathIntegralFunctional(sobolev, paths[idx], n_quad=300))
    for idx in repr_indices
]
titles_rep = [f"y={y_positions[idx]:.2f}" for idx in repr_indices]

# Interactive z-slice comparison across all 4 representers
fig = compare_function3d_axis_sliders(
    representers,
    axis="z",
    n_slices=40,
    resolution=60,
    titles=titles_rep,
    colorscale="RdBu_r",
)


---
## 7. End-to-end minimum-norm inversion

We now solve the inverse problem: given measurements $\mathbf{d} = Gu + \boldsymbol{\varepsilon}$ find the **minimum-norm** solution $\hat{u}$ that fits the data, using `pygeoinf`'s `LinearMinimumNormInversion`.

The minimum-norm solution lives in the Sobolev space (smooth representer expansion), so $G^\dagger$ calls `Sobolev3D.from_dual` internally.

In [ ]:
from pygeoinf import LinearForwardProblem, LinearMinimumNormInversion, EigenSolver

# Rebuild forms on the spectral Sobolev space defined in section 6.
forms = [PathIntegralFunctional(sobolev, p, n_quad=300) for p in paths]
G_sob = SOLAOperator3D(sobolev, EuclideanSpace(n_paths), forms)

# Synthetic data (noiseless)
d_sob = G_sob(u_true)

# Inversion
forward_problem = LinearForwardProblem(G_sob)
inversion = LinearMinimumNormInversion(forward_problem)
inverse_op = inversion.minimum_norm_operator(
    EigenSolver(galerkin=True, rtol=1e-12),
    minimum_damping=1e-6,
)
u_hat = inverse_op(d_sob)

residual = np.linalg.norm(G_sob(u_hat) - d_sob)
print(f"Residual ‖Gu_hat - d‖ = {residual:.3e}")


In [ ]:
# Compare true model vs recovered model at z = 0.5 slice
res = 120
z0 = 0.5
xg = np.linspace(0.0, 1.0, res)
yg = np.linspace(0.0, 1.0, res)
XX, YY = np.meshgrid(xg, yg, indexing="xy")
pts_grid = np.column_stack([XX.ravel(), YY.ravel(), np.full(XX.size, z0)])

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)

vals_true = u_true(pts_grid).reshape(res, res)
vals_hat = u_hat(pts_grid).reshape(res, res)
vmax = max(np.abs(vals_true).max(), np.abs(vals_hat).max())

for ax, vals, title in zip(
    axes,
    [vals_true, vals_hat],
    ["True model $u$", "Minimum-norm solution $\hat{u}$"],
):
    im = ax.contourf(XX, YY, vals, levels=30, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    for p in paths:
        t_p = np.linspace(0, 1, 100)
        c_p = p(t_p)
        ax.plot(c_p[:, 0], c_p[:, 1], "k--", lw=0.8, alpha=0.5)
    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(f"{title}\n(z = {z0:.1f} slice)")
    plt.colorbar(im, ax=ax, shrink=0.85)

plt.suptitle("Path-integral minimum-norm inversion (8 Sobolev paths)", fontsize=12)
plt.show()
